# 06 — Kafka Producer
**Role: Pipeline & Visualization Engineer**

Streams Grab reviews from `cleaned_data.csv` into a Kafka topic row-by-row, simulating real-time ingestion.  
Each message is a JSON payload containing the review fields.

**Pipeline position:**  
`cleaned_data.csv` → **Kafka Producer** → topic `grab_reviews` → Spark Consumer

## 1. Install dependencies

In [ ]:
# Install kafka-python if not already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kafka-python', '--quiet'])
print('kafka-python ready')

## 2. Configuration

In [ ]:
# ── Data file ──────────────────────────────────────────────────
DATA_FILE = 'cleaned_data (2).csv'
for _c in ['cleaned_data (2).csv', 'cleaned_data.csv',
           '../cleaned_data (2).csv', '../cleaned_data.csv']:
    if __import__('os').path.exists(_c):
        DATA_FILE = _c
        break

# ── Streaming parameters ───────────────────────────────────────
DELAY_SECONDS = 0.3   # pause between messages  (lower = faster)
BATCH_SIZE    = 1     # messages per send call

print(f'Bootstrap : {KAFKA_BOOTSTRAP}')
print(f'Topic     : {KAFKA_TOPIC}')
print(f'Data file : {DATA_FILE}')
print(f'Delay     : {DELAY_SECONDS}s per message')

## 3. Load data

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} rows  |  columns: {list(df.columns)}')
df.head(3)

## 4. Create Kafka topic (if it does not exist)

In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP, client_id='grab-admin')

topic_cfg = NewTopic(
    name=KAFKA_TOPIC,
    num_partitions=3,          # 3 partitions for parallel Spark tasks
    replication_factor=1       # 1 broker in local dev
)

try:
    admin.create_topics([topic_cfg])
    print(f'Topic "{KAFKA_TOPIC}" created.')
except TopicAlreadyExistsError:
    print(f'Topic "{KAFKA_TOPIC}" already exists — skipping.')
finally:
    admin.close()

## 5. Start streaming

In [ ]:
import json, time
from kafka import KafkaProducer

# ── Helper: delivery report callback ──────────────────────────────
sent_count   = 0
failed_count = 0

# ── Producer with JSON serialiser ─────────────────────────────────
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    acks='all',                       # strongest delivery guarantee
    retries=3,
    linger_ms=5,                      # small batch window
    compression_type='gzip'
)

print(f'Producer connected → {KAFKA_BOOTSTRAP}')
print(f'Streaming {len(df):,} reviews into topic "{KAFKA_TOPIC}" ...\n')

# Columns to keep in each Kafka message
KEEP_COLS = [
    'review_id', 'username', 'timestamp', 'final_clean_text',
    'score', 'sentiment_category', 'sentiment_label',
    'confidence_level', 'app_version', 'thumbs_up_count',
    'word_count', 'has_reply'
]
# Keep only columns that exist in the dataframe
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]

t_start = time.time()

for idx, row in df[KEEP_COLS].iterrows():
    record = row.to_dict()
    # Convert NaN → None so JSON serialisation works
    record = {k: (None if pd.isna(v) else v) for k, v in record.items()}

    # Add an ingestion timestamp
    record['ingested_at'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

    key = str(record.get('review_id', idx))

    future = producer.send(KAFKA_TOPIC, key=key, value=record)
    sent_count += 1

    # Progress every 100 messages
    if sent_count % 100 == 0:
        elapsed = time.time() - t_start
        print(f'  Sent {sent_count:>5,} / {len(df):,}  |  {elapsed:.1f}s elapsed')

    time.sleep(DELAY_SECONDS)

# Flush remaining messages
producer.flush()
producer.close()

elapsed = time.time() - t_start
print(f'\n✅ Done — {sent_count:,} messages sent in {elapsed:.1f}s')

## 6. Quick verification — consume a few messages back

In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=5000      # stop after 5 s of no messages
)

samples = []
for msg in consumer:
    samples.append(msg.value)
    if len(samples) >= 5:
        break
consumer.close()

print(f'Sampled {len(samples)} messages from topic "{KAFKA_TOPIC}":\n')
for i, s in enumerate(samples, 1):
    print(f'[{i}] review_id={s.get("review_id")}  '
          f'score={s.get("score")}  '
          f'sentiment={s.get("sentiment_category")}  '
          f'text_snippet="{str(s.get("final_clean_text",""))[:60]}..."')

---
### Summary
| Step | Detail |
|------|--------|
| Source | `cleaned_data.csv` — pre-labelled Grab reviews |
| Kafka topic | `grab_reviews` (3 partitions) |
| Message format | JSON with all review fields + `ingested_at` timestamp |
| Next notebook | `07_spark_streaming.ipynb` — consumes this topic, loads model, sinks to Elasticsearch |